## 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

## raw 데이터 확인

In [ ]:
%%bigquery df_up_types --project your-gcp-project
-- user_properties 실제 컬럼·타입 확인 (user_id 있는지가 관건)
SELECT column_name, data_type
FROM `your-gcp-project.events.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'user_properties' -- event_properties와 다른 테이블인 점 유의!!
ORDER BY ordinal_position;

In [ ]:
df_up_types

### 매칭 여부 및 데이터 중복/결측 확인

In [ ]:
%%bigquery df_up_qc --project your-gcp-project
WITH base AS (
  SELECT
    SAFE_CAST(user_id AS INT64) AS user_id_int,
    user_id, gender, grade, class, school_id
  FROM `your-gcp-project.events.user_properties`
)
SELECT
  COUNT(*)                                   AS total_rows,
  COUNT(DISTINCT user_id_int)                AS distinct_users,     -- 유저 중복 여부
  COUNT(*) - COUNT(DISTINCT user_id_int)     AS dup_users,          -- 0이면 1유저=1행
  COUNTIF(user_id_int IS NULL)               AS null_or_nonnumeric, -- 숫자 아닌 user_id
  COUNTIF(gender IS NULL)                     AS null_gender,
  COUNTIF(grade IS NULL)                      AS null_grade,
  COUNTIF(school_id IS NULL)                  AS null_school
FROM base;

In [ ]:
df_up_qc

### 숫자가 아닌 user_id

In [ ]:
%%bigquery df_bad_userid --project your-gcp-project
-- user_id가 NULL이거나 숫자로 변환 안 되는 행 실제 값 확인
SELECT user_id, gender, grade, class, school_id
FROM `your-gcp-project.events.user_properties`
WHERE SAFE_CAST(user_id AS INT64) IS NULL   -- 변환 실패 = NULL 또는 비숫자

In [ ]:
df_bad_userid

## 값 분포
- 중복 유저 2명
- user_id가 숫자 아닌 것 2건
- 성별, 학년, 학교 결측값 없음
- ⚠️ gender가 votes에도 존재 → 세그먼트 기준 소스는 논의 후 결정 필요
> 추후 EDA 과정에서 자세히 다루기

In [ ]:
%%bigquery df_up_dist --project your-gcp-project
SELECT 'gender' AS field, CAST(gender AS STRING) AS value, COUNT(*) AS cnt
FROM `your-gcp-project.events.user_properties` GROUP BY value
UNION ALL
SELECT 'grade', CAST(grade AS STRING), COUNT(*)
FROM `your-gcp-project.events.user_properties` GROUP BY grade
UNION ALL
SELECT 'class', CAST(class AS STRING), COUNT(*)
FROM `your-gcp-project.events.user_properties` GROUP BY class
ORDER BY field, cnt DESC;

In [ ]:
df_up_dist

## Stage

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_user_properties` AS
WITH src AS (
  SELECT
    SAFE_CAST(user_id AS INT64) AS user_id,   -- STRING → INT64 (조인키 통일)
    gender, grade, class, school_id
  FROM `your-gcp-project.events.user_properties`
  WHERE REGEXP_CONTAINS(user_id, r'^[0-9]+$')  -- 비숫자 user_id 2건 제외
),
dedup AS (
  -- 유저 중복 2건 방어: user_id당 1행만 (임의 1건)
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY school_id) AS rn
  FROM src
)
SELECT
  user_id,                                     -- 조인키 (검증된 events=votes id)
  gender,                                      -- 성별 (F/M)
  grade,                                       -- 학년 (1~3)
  class,                                       -- 반
  school_id                                    -- 학교 id (축 B 연결)
FROM dedup
WHERE rn = 1;

### 생성 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*)                        AS total_users,      -- 230,817 (중복 제거 후)
  COUNT(*) - COUNT(DISTINCT user_id) AS dup_check,     -- 0이어야
  COUNTIF(user_id IS NULL)        AS null_user_id,     -- 0
  COUNT(DISTINCT school_id)       AS school_count,     -- 학교 수 (축 B 참고)
  COUNTIF(gender = 'F') AS female, COUNTIF(gender = 'M') AS male
FROM `your-gcp-project.stage.stg_user_properties`;